In [16]:
import pickle
with open('vocabs.pkl', 'rb') as f:
    vocabs = pickle.load(f)

In [17]:
len(vocabs)

10000

In [18]:
maxid = max(vocabs, key=lambda a: len(vocabs[a]))

In [19]:
vocabs[maxid]

b' accomplishment'

In [3]:
[bytes([b]) for b in 'aasdf'.encode()]

[b'a', b'a', b's', b'd', b'f']

In [1]:
from encode_decode import Tokenizer

tokenizer = Tokenizer.from_files(vocab_filepath='vocabs.pkl', merges_filepath='merges.pkl')
tokenizer.encode('ssafff')

[563, 3292, 102]

# tokenizer_experiments

In [2]:
def sample_documents(path, n=10):
    docs, current = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if "<|endoftext|>" in line:
                before, _, after = line.partition("<|endoftext|>")
                current.append(before)
                doc = "".join(current).strip()
                if doc:
                    docs.append(doc)
                    if len(docs) >= n:
                        break
                current = [after]
            else:
                current.append(line)
    return docs[:n]
sample_tiny = sample_documents('../TinyStoriesV2-GPT4-valid.txt')
sample_openweb = sample_documents('../owt_train.txt')

sample_tiny_text =" ".join(chunk for chunk in sample_tiny)
sample_openweb_text =" ".join(chunk for chunk in sample_openweb)

In [7]:
tiny_enc = tokenizer.encode(sample_tiny_text)
compression_ratio_tiny = len(list(sample_tiny_text.encode()))/len(tiny_enc)
print(f'compression ratio for tinystories sample: {compression_ratio_tiny}')

import time

sample_bytes = len(sample_tiny_text.encode("utf-8"))

start = time.perf_counter()
tokenizer.encode(sample_tiny_text)
elapsed = time.perf_counter() - start

throughput = sample_bytes / elapsed
print(f"throughput: {throughput:,.0f} bytes/sec")
pile_bytes = 825 * (1024**3)
seconds = pile_bytes / throughput
print(f"pile dataset: {seconds/3600:.1f} hours  ({seconds/86400:.2f} days)")

compression ratio for tinystories sample: 4.039039039039039
throughput: 745,647 bytes/sec
pile dataset: 330.0 hours  (13.75 days)


In [4]:
openweb_enc = tokenizer.encode(sample_openweb_text)
compression_ratio_openweb = len(list(sample_openweb_text.encode()))/len(openweb_enc)
print(f'compression ratio for openweb sample: {compression_ratio_openweb}')

compression ratio for openweb sample: 3.1724415793714744


In [3]:
import pickle
with open('vocabs.pkl', 'rb') as f:
    vocabs = pickle.load(f)

with open('merges.pkl', 'rb') as f:
    merges = pickle.load(f)

In [4]:
from encode_decode import Tokenizer

tokenizer = Tokenizer(vocabs, merges, ['<|endoftext|>'])


In [5]:
import numpy as np

with open("tt.txt") as f:
    ids = np.fromiter(tokenizer.encode_iterable(f), dtype=np.uint16)

ids.tofile("tt.bin")


In [ ]:
with open("../TinyStoriesV2-GPT4-valid.txt") as f:
    val_ids = np.fromiter(tokenizer.encode_iterable(f), dtype=np.uint16)
val_ids.tofile("val_tinystories.bin")


In [6]:
data = np.memmap("tt.bin", dtype=np.uint16, mode="r")
print(len(data), data.max(), data[:20])


210 3632 [ 431  440  259  399   44  314  259 1052  267 1169  961   44  402  283
  259  347 3632   46  439  391]
